In [1]:
!nvidia-smi

Fri Oct  9 05:05:30 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   42C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [5]:
from numba import cuda
import os
device = cuda.get_current_device()
print(f"Device Name: {device.name}")
print(f"Compute Capability: {device.compute_capability}")
print(f"Multiprocessor Count (SMs): {getattr(device, 'MULTIPROCESSOR_COUNT', 'N/A')}")
print(f"Max Threads Per Block: {device.MAX_THREADS_PER_BLOCK}")
print(f"Max Block Dimensions: {device.MAX_BLOCK_DIM_X}, {device.MAX_BLOCK_DIM_Y},{device.MAX_BLOCK_DIM_Z}")
print(f"Warp Size: {device.WARP_SIZE}")

Device Name: Tesla T4
Compute Capability: ComputeCapability(major=7, minor=5)
Multiprocessor Count (SMs): 40
Max Threads Per Block: 1024
Max Block Dimensions: 1024, 1024,64
Warp Size: 32


In [7]:
import numpy as np
import time
from numba import cuda

# 1. CPU Reference Implementation (Pure Python loop)
def double_cpu_pure(arr):
    out = np.empty_like(arr)
    for i in range(len(arr)):
        out[i] = arr[i] * 2.0
    return out

# 2. GPU Kernel Implementation
@cuda.jit
def double_gpu_kernel(d_in, d_out):
    # Calculate unique 1D thread ID
    idx = cuda.grid(1)
    if idx < d_in.shape[0]:
        d_out[idx] = d_in[idx] * 2.0

In [10]:
def benchmark_run(N, student_id_seed=1234):
    np.random.seed(student_id_seed)
    h_in = np.random.rand(N).astype(np.float32)

    if N <= 1_000_000:
        t0 = time.perf_counter()
        _ = double_cpu_pure(h_in)
        cpu_time_ms = (time.perf_counter() - t0) * 1000.0
    else:
        cpu_time_ms = float('nan')

    threads_per_block = 256
    blocks_per_grid = (N + threads_per_block - 1) // threads_per_block

    # Warmup compilation run
    d_warmup = cuda.to_device(np.ones(10, dtype=np.float32))
    double_gpu_kernel[1, 32](d_warmup, d_warmup)
    cuda.synchronize()

    # GPU roundtrip (includes PCIe transfers)
    t0 = time.perf_counter()
    d_in = cuda.to_device(h_in)
    d_out = cuda.device_array_like(d_in)
    double_gpu_kernel[blocks_per_grid, threads_per_block](d_in, d_out)
    h_out = d_out.copy_to_host()
    cuda.synchronize()
    gpu_total_ms = (time.perf_counter() - t0) * 1000.0

    # GPU compute only (data already in VRAM)
    t0 = time.perf_counter()
    double_gpu_kernel[blocks_per_grid, threads_per_block](d_in, d_out)
    cuda.synchronize()
    gpu_compute_ms = (time.perf_counter() - t0) * 1000.0

    return cpu_time_ms, gpu_total_ms, gpu_compute_ms

In [25]:
for N in [100, 1_000, 10_000, 100_000, 1_000_000, 5_000_000, 10_000_000]:
    cpu, total, kernel = benchmark_run(N)
    print(f"{N:>10,} | CPU {cpu:10.3f} | GPU total {total:10.3f} | kernel {kernel:10.3f}")

       100 | CPU      0.101 | GPU total      2.049 | kernel      0.143
     1,000 | CPU      0.589 | GPU total      0.788 | kernel      0.113
    10,000 | CPU      5.677 | GPU total      0.796 | kernel      0.119
   100,000 | CPU     55.482 | GPU total      0.893 | kernel      0.105
 1,000,000 | CPU    521.598 | GPU total      5.529 | kernel      0.146
 5,000,000 | CPU        nan | GPU total     18.294 | kernel      0.297
10,000,000 | CPU        nan | GPU total     25.985 | kernel      0.462


In [14]:
def benchmark_run(N, student_id_seed=1234):
    np.random.seed(student_id_seed)
    h_in = np.random.rand(N).astype(np.float32)
    # Measure CPU Pure Loop (skip for N >= 10M if too slow)
    if N <= 1_000_000:
        t0 = time.perf_counter()
        _ = double_cpu_pure(h_in)
        cpu_time_ms = (time.perf_counter() - t0) * 1000.0
    else:
        cpu_time_ms = float('nan') # Extrapolate or skip
    # GPU Setup
    threads_per_block = 256
    blocks_per_grid = (N + threads_per_block - 1) // threads_per_block
    # Warmup compilation run
    d_warmup = cuda.to_device(np.ones(10, dtype=np.float32))
    double_gpu_kernel[1, 32](d_warmup, d_warmup)
    cuda.synchronize()
    # GPU Roundtrip Measurement (Includes PCIe transfers)
    t0 = time.perf_counter()
    d_in = cuda.to_device(h_in)
    d_out = cuda.device_array_like(d_in)
    double_gpu_kernel[blocks_per_grid, threads_per_block](d_in, d_out)
    h_out = d_out.copy_to_host()
    cuda.synchronize()
    gpu_total_ms = (time.perf_counter() - t0) * 1000.0
    # GPU Compute-Only Measurement (Pre-allocated in VRAM)
    t0 = time.perf_counter()
    double_gpu_kernel[blocks_per_grid, threads_per_block](d_in, d_out)
    cuda.synchronize()
    gpu_compute_ms = (time.perf_counter() - t0) * 1000.0
    return cpu_time_ms, gpu_total_ms, gpu_compute_ms

In [16]:
# DEFECTIVE KERNEL: Notice the missing 'if idx < n' check!
@cuda.jit
def defective_kernel(d_arr):
    idx = cuda.grid(1)
    # FORBIDDEN: Writing without bounds check!
    d_arr[idx] = 999.0

# TEST HARNESS
N = 1000
h_arr = np.zeros(N, dtype=np.float32)
d_arr = cuda.to_device(h_arr)

# Launch 4 blocks of 256 = 1024 threads
threads_per_block = 256
blocks_per_grid = 4

print("Launching defective kernel with 1024 threads on 1000-element array...")
defective_kernel[blocks_per_grid, threads_per_block](d_arr)
cuda.synchronize()

print("Execution finished (or did it crash?). Checking host copy...")
h_result = d_arr.copy_to_host()
print(f"Elements processed: {len(h_result)}. Last element: {h_result[-1]}")

Launching defective kernel with 1024 threads on 1000-element array...
Execution finished (or did it crash?). Checking host copy...
Elements processed: 1000. Last element: 999.0


/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 4 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))


In [18]:
@cuda.jit
def inspect_threads_kernel(log_block, log_thread, log_global, log_active, N):
    t_idx = cuda.threadIdx.x
    b_idx = cuda.blockIdx.x
    g_idx = cuda.grid(1)
    if g_idx < log_block.shape[0]:
        log_block[g_idx] = b_idx
        log_thread[g_idx] = t_idx
        log_global[g_idx] = g_idx
        log_active[g_idx] = 1 if g_idx < N else 0

# Configuration
N = 37
threads_per_block = 8
blocks_per_grid = (N + threads_per_block - 1) // threads_per_block # Exactly 5 blocks = 40 threads

log_b = cuda.device_array(blocks_per_grid * threads_per_block, dtype=np.int32)
log_t = cuda.device_array(blocks_per_grid * threads_per_block, dtype=np.int32)
log_g = cuda.device_array(blocks_per_grid * threads_per_block, dtype=np.int32)
log_a = cuda.device_array(blocks_per_grid * threads_per_block, dtype=np.int32)

inspect_threads_kernel[blocks_per_grid, threads_per_block](log_b, log_t, log_g, log_a, N)
cuda.synchronize()

print(f"{'Global ID':<10} | {'Block ID':<10} | {'Thread ID':<10} | {'Status':<10}")
print("-" * 50)
for i in range(32, 40): # Print the critical boundary transition (Block 4)
    status = "ACTIVE" if log_a[i] == 1 else "IDLE (GUARDED)"
    print(f"{log_g[i]:<10} | {log_b[i]:<10} | {log_t[i]:<10} | {status:<10}")

Global ID  | Block ID   | Thread ID  | Status    
--------------------------------------------------
32         | 4          | 0          | ACTIVE    
33         | 4          | 1          | ACTIVE    
34         | 4          | 2          | ACTIVE    
35         | 4          | 3          | ACTIVE    
36         | 4          | 4          | ACTIVE    
37         | 4          | 5          | IDLE (GUARDED)
38         | 4          | 6          | IDLE (GUARDED)
39         | 4          | 7          | IDLE (GUARDED)


/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 5 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))


In [26]:
d_arr = cuda.to_device(np.ones(50000, dtype=np.float32))
tpb = 256
bpg = (50000 + tpb - 1) // tpb

for k in (naive_sum_kernel, atomic_sum_kernel):   # warm-up compile
    k[bpg, tpb](d_arr, cuda.to_device(np.zeros(1, dtype=np.float32)))
cuda.synchronize()

for trial in range(1, 6):
    d_total = cuda.to_device(np.zeros(1, dtype=np.float32))
    t0 = time.perf_counter()
    naive_sum_kernel[bpg, tpb](d_arr, d_total); cuda.synchronize()
    ms = (time.perf_counter() - t0) * 1000
    val = d_total.copy_to_host()[0]
    print(f"Trial {trial}: {val} | lost = {50000 - val} | {ms:.3f} ms")

d_total = cuda.to_device(np.zeros(1, dtype=np.float32))
t0 = time.perf_counter()
atomic_sum_kernel[bpg, tpb](d_arr, d_total); cuda.synchronize()
print("Atomic:", d_total.copy_to_host()[0], f"{(time.perf_counter()-t0)*1000:.3f} ms")

Trial 1: 2.0 | lost = 49998.0 | 0.175 ms
Trial 2: 2.0 | lost = 49998.0 | 0.165 ms
Trial 3: 2.0 | lost = 49998.0 | 0.118 ms
Trial 4: 2.0 | lost = 49998.0 | 0.119 ms
Trial 5: 2.0 | lost = 49998.0 | 0.106 ms
Atomic: 50000.0 0.474 ms


In [20]:
# BUGGY NAIVE SUMMATION
@cuda.jit
def naive_sum_kernel(d_arr, d_total):
    idx = cuda.grid(1)
    if idx < d_arr.shape[0]:
        # DATA RACE: Thousands of threads execute Read-Modify-Write at the exact same clock cycle!
        d_total[0] += d_arr[idx]

# CORRECT ATOMIC SUMMATION
@cuda.jit
def atomic_sum_kernel(d_arr, d_total):
    idx = cuda.grid(1)
    if idx < d_arr.shape[0]:
        # HARDWARE ATOMIC LOCK: Hardware serializes updates to prevent collisions
        cuda.atomic.add(d_total, 0, d_arr[idx])

In [22]:
import math
@cuda.jit
def radial_vignette_2d(d_canvas, width, height, center_x, center_y):
    # Retrieve 2D spatial coordinates directly from the grid
    x, y = cuda.grid(2)
    if x < width and y < height:
        dx = float(x - center_x)
        dy = float(y - center_y)
        dist = math.sqrt(dx * dx + dy * dy)
        max_dist = math.sqrt(float(center_x**2 + center_y**2))
        intensity = 1.0 - (dist / max_dist)
        if intensity < 0.0:
            intensity = 0.0
        d_canvas[y, x] = intensity

# 2D EXECUTION CONFIGURATION
WIDTH, HEIGHT = 1024, 1024
canvas = np.zeros((HEIGHT, WIDTH), dtype=np.float32)
d_canvas = cuda.to_device(canvas)

# 2D Blocks (16x16 = 256 threads per block)
threads_2d = (16, 16)
blocks_2d = (
    (WIDTH + threads_2d[0] - 1) // threads_2d[0],
    (HEIGHT + threads_2d[1] - 1) // threads_2d[1]
)

radial_vignette_2d[blocks_2d, threads_2d](d_canvas, WIDTH, HEIGHT, WIDTH // 2, HEIGHT // 2)
cuda.synchronize()
result_canvas = d_canvas.copy_to_host()
print(f"Generated 2D matrix shape: {result_canvas.shape}. Center value: {result_canvas[512,  512]:.3f}")

Generated 2D matrix shape: (1024, 1024). Center value: 1.000


In [24]:
import hashlib
def generate_lab_checksum(student_id_str, d_canvas, exp_table_crossover):
 h = hashlib.sha256()
 h.update(str(student_id_str).strip().encode('utf-8'))
 # Include GPU name (ensuring it is bytes)
 gpu_name = cuda.get_current_device().name
 if isinstance(gpu_name, str):
  gpu_name = gpu_name.encode('utf-8')
 h.update(gpu_name)
 # Include sample of compute output
 h.update(d_canvas.copy_to_host()[:10, :10].tobytes())
 h.update(str(exp_table_crossover).encode('utf-8'))
 digest = h.hexdigest()[:16].upper()
 print("=" * 60)
 print(f"OFFICIAL VERIFICATION CHECKSUM TOKEN: {digest}")
 print("=" * 60)
 return digest
# REPLACE WITH YOUR ACTUAL STUDENT ID AND EMPIRICAL CROSSOVER N* VALUE
STUDENT_ID = 230107088
CROSSOVER_N = 10000 # Example: your value from Section 2
generate_lab_checksum(STUDENT_ID, d_canvas, CROSSOVER_N)

OFFICIAL VERIFICATION CHECKSUM TOKEN: F81F12D38F09EF7E


'F81F12D38F09EF7E'